# Spam and Phishing Email Detection using NLP
**Dataset:** [SetFit/enron_spam](https://huggingface.co/datasets/SetFit/enron_spam) - 33,716 labelled emails (ham / spam).

## Terminology
* **Corpus** - A collection of documents. Our corpus is the full SetFit/enron_spam dataset
* **Token** - A single word produced after tokenization
* **Stopwords** - Extremely common words that carry little discriminative meaning. Example: This, is, and, or
* **Stemming** - chopping words down to a crude root (winning - win, flies - fli)
* **Lemmatization** - Reducing words to their dictionary form (winning -> win, flies -> fly)
* **TD-IDF(Term Frequency-Inverse Document Frequency)** - Is this word used a lot here, but rarely everywhere else?"  if yes, it's probably important.
* **N-Gram** - "What words appear together in this document?"Because in real language, context matters  "free" is innocent, but "free money now" is suspicious.
* **Embedding** - Instead of just counting words, embeddings understand that "free prize" and "complimentary reward" are saying the same suspicious thing
## Models compared

| # | Model | Feature representation | Family |
|---|-------|------------------------|--------|
| 1 | Support Vector Machine     | TF-IDF (unigram + bigram) | Classical ML |
| 2 | Random Forest              | TF-IDF (unigram + bigram) | Classical ML |
| 3 | DistilBERT (fine-tuned)    | Sub-word embeddings       | Transformer |

## 1. Importing Libraries

In [ ]:
import re # finding and cleaning patterns
import  warnings # ignoring warnings
import random # random choices
from collections import Counter # counting tokens

# Data and visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP tools
import nltk
from nltk.corpus import stopwords # Common word with no meaning
from nltk.stem import WordNetLemmatizer, PorterStemmer # Stemming and Lemmatization

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer # Feature extraction
from sklearn.svm import LinearSVC # Support Vector Machine
from sklearn.ensemble import RandomForestClassifier # Random Forest
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve,
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid') # Make plot look nice
plt.rcParams['figure.dpi'] = 100 # Controls plot resolution

SEED = 42
random.seed(SEED)  # Python random module
np.random.seed(SEED) # NumPy random module

# Downloads 5 required NLTK data packages
for pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f'[nltk] skip {pkg}: {e}')

print('All libraries imported successfully.')


## 2. Load Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('SetFit/enron_spam')

train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

# Merging both train and test spilted data
df =pd.concat([train_df, test_df], ignore_index=True)

# Give every column to be the correct type
df['subject'] = df['subject'].fillna('').astype(str)
df['text'] = df['text'].fillna('').astype(str)
df['label'] = df['label'].astype(int)
df['label_text'] = df['label_text'].astype(str)
print()
print('Dataset shape :', df.shape)
print('Columns Names     :', df.columns.tolist())
print()
df.head(3)


## 3. Exploratory Data Analysis


In [ ]:
print(df.info())

In [ ]:
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Class distribution (count):')
print(df['label_text'].value_counts())
print()
print('Class distribution (%):')
print(df['label_text'].value_counts(normalize=True).mul(100).round(2))




### EDA 1: Class balance (bar + pie)

In [ ]:
#Creating two plot and width, height
fig, axes = plt.subplots(1, 2, figsize=(12,5))
#Seperating two tables
counts = df['label_text'].value_counts()

# Bar Chart
bars = axes[0].bar(counts.index, counts.values, color=['#4C9AFF', '#FF6B6B'], edgecolor='white')
axes[0].set_title('Class Distribution - Count')
axes[0].set_ylabel('Number of emails')
axes[0].bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=10)

#Pie Chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4C9AFF', '#FF6B6B'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution - Proportion')
plt.tight_layout();
plt.show()


### EDA 2: Document length (Chars and Words)

In [ ]:
df['char_len'] = df['text'].str.len()
df['word_len'] = df['text'].str.split().str.len()

fig,axes = plt.subplots(1,2 , figsize=(13,5))
# Hist Plot
sns.histplot(data=df, x='word_len', hue='label_text', bins=100,
             ax=axes[0], log_scale=False,  # ← remove log scale
             palette={'ham': '#4C9AFF', 'spam': '#FF6B6B'})
axes[0].set_xlim(0, 500)  # ← zoom into where data actually is
axes[0].set_title('Word-count distribution (log-scaled)')
axes[0].set_xlim(0, 2000); axes[0].set_xlabel('Word count')

#Box Plot
sns.boxplot(data=df, x='label_text', y='word_len', ax=axes[1],
            palette={'ham': '#4C9AFF', 'spam': '#FF6B6B'})
axes[1].set_title('Word count per class')
axes[1].set_ylim(0, 2000); axes[1].set_xlabel('Class')

plt.tight_layout();
plt.show()

print(df.groupby('label_text')['word_len'].describe().round(1))

###  EDA 3: Character-length density per class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for lbl, color in [('ham', '#4C9AFF'), ('spam', '#FF6B6B')]:
    sns.kdeplot(df.loc[df['label_text'] == lbl, 'char_len'].clip(upper=8000),
                ax=ax, fill=True, alpha=0.4, color=color, label=lbl)
ax.set_title('Character-length density by class')
ax.set_xlabel('Characters per email (clipped at 8000)')
ax.legend();
plt.show()

###  EDA 4: Word clouds per class

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, lbl, cmap in [(axes[0], 'ham', 'Blues'), (axes[1], 'spam', 'Reds')]:
        sample = df[df['label_text'] == lbl]['text'] \
                    .sample(min(2000, (df['label_text'] == lbl).sum()), random_state=SEED)
        wc = WordCloud(width=600, height=350, background_color='white',
                       max_words=100, colormap=cmap)
        wc.generate(' '.join(sample))
        ax.imshow(wc); ax.axis('off')
        ax.set_title(f'{lbl.upper()} - top vocabulary', fontsize=14)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('WordCloud skipped:', e)

###  EDA 5: Top-20 frequent tokens per class

In [ ]:
def top_tokens(series, n=20):
    counter = Counter()
    for t in series.sample(min(5000, len(series)), random_state=SEED):
        counter.update(re.findall(r'[a-zA-Z]{3,}', str(t).lower()))
    return counter.most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, lbl, color in [(axes[0], 'ham', '#4C9AFF'), (axes[1], 'spam', '#FF6B6B')]:
    pairs = top_tokens(df.loc[df['label_text'] == lbl, 'text'], 20)
    words, counts = zip(*pairs)
    ax.barh(words, counts, color=color)
    ax.invert_yaxis()
    ax.set_title(f'Top 20 words - {lbl.upper()}')
    ax.set_xlabel('Frequency')
plt.tight_layout();
plt.show()

###  EDA 6: Correlation between numeric features and label

In [ ]:
numeric = df[['char_len', 'word_len', 'label']].copy()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(numeric.corr(), annot=True, cmap='vlag', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation: numeric features vs. label')
plt.show()

###  EDA 7: Subject vs body length contribution

In [ ]:
df['subject_len'] = df['subject'].str.split().str.len()
df['body_len']    = df['text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(9, 4))
sub = df.groupby('label_text')[['subject_len', 'body_len']].mean()
sub.plot(kind='bar', ax=ax, color=['#26A69A', '#FFA726'], edgecolor='white')
ax.set_title('Mean subject vs body length by class')
ax.set_ylabel('Mean word count'); ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout(); plt.show()
print(sub.round(2))


## 4. Text Processing
This section builds the exact cleaning pipeline used for the classical models (SVM and Random Forest).

What the code does:
- lower-cases text
- removes HTML tags and URLs
- removes non-letter characters and extra spaces
- tokenises by whitespace
- drops very short tokens
- removes stopwords
- applies **lemmatization**

Then the pipeline is applied to each email to create `clean_text`, which is used by TF-IDF.

> DistilBERT does not use this cleaning pipeline; it uses raw text.

In [ ]:
# ── Regex Patterns

HTML_RE     = re.compile(r'<[^>]+>')                        # matches <any html tag>
URL_RE      = re.compile(r'(https?://\S+|www\.\S+)',        # matches http:// or www. links
                         flags=re.IGNORECASE)               # case-insensitive (HTTP:// also caught)
EMOJI_RE    = re.compile(
    '['
    '\U0001F600-\U0001F64F'   # emoticons
    '\U0001F300-\U0001F5FF'   # symbols & pictographs
    '\U0001F680-\U0001F6FF'   # transport & map symbols
    '\U0001F1E0-\U0001F1FF'   # flags
    '\U00002702-\U000027B0'
    '\U000024C2-\U0001F251'
    ']+', flags=re.UNICODE)
NON_ALPHA   = re.compile(r'[^a-z\s]')                       # matches anything that is NOT a letter or space
MULTI_SPACE = re.compile(r'\s+')                            # matches multiple consecutive spaces/tabs

# ── Stopwords
STOP_WORDS = set(stopwords.words('english'))                # standard English stopwords (the, is, a ...)
STOP_WORDS.update(['subject', 'enron', 'http',             # add dataset-specific noise words
                   'www', 'com', 'org'])                    # these appear everywhere and carry no signal

# ── Lemmatizer
LEMMATIZER = WordNetLemmatizer()                            # converts words to root form (running → run)


# ── Helper Functions

def remove_html(text: str) -> str:
    """Strip HTML tags — emails frequently contain inline HTML."""
    return HTML_RE.sub(' ', text)              # replace tag with space, not empty string


def remove_urls(text: str) -> str:
    """Remove all URLs — raw links add noise, not signal."""
    return URL_RE.sub(' ', text)              # replace URL with space

def remove_emoji(text: str) -> str:
    """Remove emoji characters from email text (used in the demo only)."""
    return EMOJI_RE.sub(' ', text)

def remove_stopwords(tokens):
    """Filter out stopwords from token list."""
    return [t for t in tokens              # keep token only if
            if t not in STOP_WORDS]        # it is NOT in stopwords list


def lemmatization(tokens):
    """Reduce each token to its dictionary root form."""
    return [LEMMATIZER.lemmatize(t)        # e.g. "studies" → "study"
            for t in tokens]              #      "running"  → "run"

### 4.1 Demonstrate Step-by-step text cleaning on one example email

In [ ]:
# Synthetic noisy email to make each cleaning step visible
demo_email = (
    'Subject: WIN a FREE iPhone 15 NOW!!! '
    '<b>Click</b> https://bit.ly/free-iphone or visit www.fake-prize.com '
    'to claim your $1000 reward 🎉🎁💰. '
    'Limited offer - reply YES today!! '
    'Contact us at winner@fake.com for more info.'
)
 # A function to print before/after results of each cleaning step
def show(step, before, after, note=''):
    print(f'--- {step} ---')
    if note:
        print(f'  effect : {note}')
    print(f'  before : {before}')
    print(f'  after  : {after}')

print('Prints the original uncleaned email')
print(' ', demo_email)

#### 1. Lower Case


In [ ]:
step1 = demo_email.lower()
show('STEP 1 - lower-case', demo_email, step1,
     note='WIN -> win, FREE -> free (so they match the same token)')

#### 2. Remove HTML tags

In [ ]:
step2 = remove_html(step1)
show('STEP 2 - remove HTML tags', step1, step2,
     note='<b>...</b> stripped out')

#### 3. Remove URLs

In [ ]:
step3 = remove_urls(step2)
show('STEP 3 - remove URLs', step2, step3,
     note='https://bit.ly/... and www.fake-prize.com removed')

#### 4. Remove emoji (demo only - emojis are kept in the real pipeline)


In [ ]:
step4 = remove_emoji(step3)
show('STEP 4 - remove emoji', step3, step4,
     note='strips pictographs like 🎉🎁💰  (NOTE: not used in text_cleaning_pipeline)')

#### 5. Strip digits / Punctuation / Extra spaces


In [ ]:
step5a = NON_ALPHA.sub(' ', step4)
step5  = MULTI_SPACE.sub(' ', step5a).strip()
show('STEP 5 - strip digits / punctuation / extra spaces', step4, step5,
     note='numbers, $, !, @ and stray symbols replaced with spaces')

#### 6. Tokenise + Drop tokens shorter than 3 chars

In [ ]:

tokens = step5.split()
tokens_kept   = [t for t in tokens if len(t) > 2]
dropped_short = [t for t in tokens if len(t) <= 2]
print('--- STEP 6 - tokenise + drop tokens shorter than 3 chars ---')
print('  effect : split into words, then drop very short tokens')
print(f'  tokens before : {tokens}')
print(f'  dropped short : {dropped_short}')
print(f'  tokens after  : {tokens_kept}')

####  7. Remove Stop-Words

In [ ]:
tokens_no_stop = remove_stopwords(tokens_kept)
removed_stops  = sorted(set(tokens_kept) - set(tokens_no_stop))
print('--- STEP 7 - remove stop-words ---')
print('  effect : drop common low-signal words (the, is, your, ...)')
print(f'  removed : {removed_stops}')
print(f'  kept    : {tokens_no_stop}')

#### 8. Lemmatise

In [ ]:
tokens_lem = lemmatization(tokens_no_stop)
changed = [(b, a) for b, a in zip(tokens_no_stop, tokens_lem) if b != a]
print('--- STEP 8 - lemmatise ---')
print('  effect : reduce words to dictionary form (winning -> win)')
print(f'  before : {tokens_no_stop}')
print(f'  after  : {tokens_lem}')
print(f'  changed: {changed if changed else "no morphological changes in this sample"}')

### 4.2 Chain every helper into a single pipeline

In [ ]:
def text_cleaning_pipeline(text: str) -> str:
    """Full classical-ML cleaning pipeline using lemmatization."""
    if not isinstance(text, str):
        return ''

    text = text.lower()
    text = remove_html(text)
    text = remove_urls(text)
    text = NON_ALPHA.sub(' ', text)
    text = MULTI_SPACE.sub(' ', text).strip()

    tokens = [t for t in text.split() if len(t) > 2]
    tokens = remove_stopwords(tokens)
    tokens = lemmatization(tokens)

    return ' '.join(tokens)


print('raw      :', demo_email)
print()
print('cleaned  :', text_cleaning_pipeline(demo_email))

### 4.3 Apply the pipeline to the full corpus

In [ ]:
df['raw_text']   = (df['subject'] + ' ' + df['text']).str.strip()
df['clean_text'] = df['raw_text'].apply(text_cleaning_pipeline)

print(f'Documents cleaned : {len(df):,}')
print(f'Empty after clean : {(df["clean_text"].str.len() == 0).sum():,}')
print()
print('Showing the first 3 documents - raw vs cleaned (truncated to 200 chars):')
df[['label_text', 'raw_text', 'clean_text']].head(3)

### 4.4 Did our cleaning pipeline actually work?
Yes, Cleaning reduced average email length by 58% (339 → 143 words) and vocabulary
size by 15% (62,347 → 53,205 tokens), confirming the pipeline successfully
stripped noise while preserving meaningful signal for model training.

In [ ]:
# Diagnostic 1 + 2: length and vocabulary reduction (sample 5,000 docs for speed)
sample_idx = df.sample(min(5000, len(df)), random_state=SEED).index
raw_words   = df.loc[sample_idx, 'raw_text'].str.split()
clean_words = df.loc[sample_idx, 'clean_text'].str.split()

raw_lens   = raw_words.str.len()
clean_lens = clean_words.str.len()

raw_vocab   = set()
clean_vocab = set()
for tokens in raw_words:    raw_vocab.update(t.lower()  for t in tokens if isinstance(t, str))
for tokens in clean_words:  clean_vocab.update(tokens)

stats = pd.DataFrame({
    'metric': [
        'mean words per document',
        'median words per document',
        'unique vocabulary (5k sample)',
    ],
    'raw':     [round(raw_lens.mean(),   1), int(raw_lens.median()),   f'{len(raw_vocab):,}'],
    'cleaned': [round(clean_lens.mean(), 1), int(clean_lens.median()), f'{len(clean_vocab):,}'],
})
print('Cleaning impact (5,000-document sample):')
print(stats.to_string(index=False))
print()

# Bar chart - vocabulary reduction
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(['raw', 'cleaned'], [raw_lens.mean(), clean_lens.mean()],
            color=['#FF6B6B', '#4C9AFF'], edgecolor='white')
axes[0].set_title('Mean words per document (sample)')
axes[0].set_ylabel('Words')
for i, v in enumerate([raw_lens.mean(), clean_lens.mean()]):
    axes[0].text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=11)

axes[1].bar(['raw', 'cleaned'], [len(raw_vocab), len(clean_vocab)],
            color=['#FF6B6B', '#4C9AFF'], edgecolor='white')
axes[1].set_title('Unique vocabulary size (sample)')
axes[1].set_ylabel('Unique tokens')
for i, v in enumerate([len(raw_vocab), len(clean_vocab)]):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=11)
plt.tight_layout(); plt.show()



### 4.5 Diagnostic 3: side-by-side raw vs cleaned for one ham and one spam email
By comparing raw vs cleaned text for one ham and one spam email, we can
see that stopwords, URLs, HTML, symbols and boilerplate are successfully removed —
leaving only the core meaningful words that the model will train on.**bold text**


In [ ]:
# Diagnostic 3: side-by-side raw vs cleaned for one ham and one spam email
def _trim(t, n=260):
    t = ' '.join(str(t).split())
    return t if len(t) <= n else t[:n] + '...'

ham_row  = df[df['label_text'] == 'ham' ].sample(1, random_state=SEED).iloc[0]
spam_row = df[df['label_text'] == 'spam'].sample(1, random_state=SEED).iloc[0]

print('=' * 90)
print('HAM example')
print('-' * 90)
print('RAW    :', _trim(ham_row['raw_text']))
print()
print('CLEAN  :', _trim(ham_row['clean_text']))
print()
print('=' * 90)
print('SPAM example')
print('-' * 90)
print('RAW    :', _trim(spam_row['raw_text']))
print()
print('CLEAN  :', _trim(spam_row['clean_text']))
print('=' * 90)

## 5. Pre-processing for ML (Train / Test Split)

In [ ]:
# ── Train / Test Split

# Split row indices (not data directly) so every model trains/tests on identical rows
train_idx, test_idx = train_test_split(
    df.index,                          # split row numbers, not actual data
    test_size=0.2,                     # 80% train, 20% test
    stratify=df['label'].to_numpy(),   # keep spam/ham ratio similar in both splits.
    random_state=SEED,
)

# Select features (email text) using the split indices
X_train = df.loc[train_idx, 'clean_text'].astype(str).to_numpy(dtype=object)
X_test  = df.loc[test_idx,  'clean_text'].astype(str).to_numpy(dtype=object)

# Select labels (0=ham, 1=spam) using the same split indices
y_train = df.loc[train_idx, 'label'].to_numpy()
y_test  = df.loc[test_idx,  'label'].to_numpy()

# Confirm split sizes and that spam ratio is balanced in both sets
print(f'Train size  : {X_train.shape[0]:,}')       # should be ~80% of total
print(f'Test  size  : {X_test.shape[0]:,}')        # should be ~20% of total
print(f'Train spam %: {y_train.mean()*100:.1f}%')  # spam ratio in training set
print(f'Test  spam %: {y_test.mean()*100:.1f}%')   # should match train spam % ✅

## 6. Feature Extraction: TF-IDF
Why we use TF-IDF:

- It gives less importance to very common words (like "the", "is")
- It gives more importance to useful words (like "free", "win", "click" in spam)
- It also looks at two-word combinations (like "click here"), not just single words

Main settings (in simple terms):

- `ngram_range = (1,2)` -> Use single words + pairs of words
- `max_features = 20000` -> Keep only the top 20,000 important words/features
- `min_df = 2` -> Ignore words that appear only once (they are not useful)
- `sublinear_tf = True` -> If a word repeats many times, do not give it too much extra importance

In [ ]:
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print('TF-IDF train matrix:', X_train_tfidf.shape)
print('TF-IDF test  matrix:', X_test_tfidf.shape)
print(f'Vocabulary size    : {len(tfidf.vocabulary_):,}')


## 7. Modeling: Classical ML

Two classical models are trained on TF-IDF features:
- **SVM** (strong and fast baseline for text classification)
- **Random Forest** (ensemble baseline with feature importance)




In [ ]:
# This Function is to show report of each model
def evaluate(name, y_true, y_pred, y_prob=None):
    """Compute & pretty-print Acc / Prec / Rec / F1 / AUC for a single model."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan

    print(f'\n=== {name} ===')
    print(f'Accuracy : {acc:.4f}   Precision: {prec:.4f}   '
          f'Recall: {rec:.4f}   F1: {f1:.4f}   AUC: {auc:.4f}')
    print(classification_report(y_true, y_pred, target_names=['ham', 'spam']))

    return {
        'Model': name, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1': f1, 'AUC': auc,
        'y_pred': y_pred, 'y_prob': y_prob,
    }


results = []


### Model 1 - SVM (LinearSVC + sigmoid calibration so we can get probabilities for AUC / ROC)


In [ ]:
svm = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=2000),
    cv=2, method='sigmoid',
)
svm.fit(X_train_tfidf, y_train)
results.append(evaluate(
    'SVM (TF-IDF)',
    y_test,
    svm.predict(X_test_tfidf),
    svm.predict_proba(X_test_tfidf)[:, 1],
))


### Model 2 - Random Forest (non-linear ensemble baseline that also gives feature importance)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    max_depth=50,
    n_jobs=-1,
    random_state=SEED,
)
rf.fit(X_train_tfidf, y_train)
results.append(evaluate(
    'Random Forest (TF-IDF)',
    y_test,
    rf.predict(X_test_tfidf),
    rf.predict_proba(X_test_tfidf)[:, 1],
))

In [ ]:
# Top 20 most important words/phrases used by the Random Forest to detect spam
feat_df = pd.DataFrame({
    'feature':    tfidf.get_feature_names_out(),
    'importance': rf.feature_importances_,
})
top20 = feat_df.nlargest(20, 'importance')

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20['feature'], top20['importance'], color='#FF6B6B', edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Feature importance')
ax.set_title('Top 20 TF-IDF features (Random Forest)')
plt.tight_layout(); plt.show()

print(top20.to_string(index=False))


## 8. Modeling: DistilBERT Fine-tuning

In this phase, **DistilBERT** is fine-tuned using the **raw email text** .

Why raw text is used:
- transformer models handle context directly
- no manual cleaning is needed





In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: no GPU detected - DistilBERT training will be slow.')
    print('In Google Colab go to: Runtime -> Change runtime type -> GPU.')


In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use the EXACT same train / test indices as the classical pipeline (no leakage)
train_hf = Dataset.from_dict({
    'text':  df.loc[train_idx, 'raw_text'].tolist(),
    'label': df.loc[train_idx, 'label'].tolist(),
})
test_hf  = Dataset.from_dict({
    'text':  df.loc[test_idx,  'raw_text'].tolist(),
    'label': df.loc[test_idx,  'label'].tolist(),
})

# Sub-sample for free-tier hardware. Comment these two lines for full training.
TRAIN_SUBSAMPLE = 6_000
TEST_SUBSAMPLE  = 1_500
train_hf = train_hf.shuffle(seed=SEED).select(range(min(TRAIN_SUBSAMPLE, len(train_hf))))
test_hf  = test_hf .shuffle(seed=SEED).select(range(min(TEST_SUBSAMPLE,  len(test_hf))))

# Tokenise: max_length=128 covers ~97 % of Enron emails; 256 doubles RAM for no gain
def tok_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

train_tok = train_hf.map(tok_fn, batched=True, remove_columns=['text'])
test_tok  = test_hf.map (tok_fn, batched=True, remove_columns=['text'])
collator  = DataCollatorWithPadding(tokenizer=tokenizer)

print(f'Train samples : {len(train_tok):,}')
print(f'Test  samples : {len(test_tok):,}')


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir='./distilbert_spam',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=(device == 'cuda'),
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    report_to='none',
    dataloader_pin_memory=False,
    seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds),
    }

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=test_tok,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)

print('Starting DistilBERT fine-tuning...')
trainer.train()
print('Training complete.')


In [ ]:
# Evaluate DistilBERT on the held-out test set
pred_out = trainer.predict(test_tok)
raw      = pred_out.predictions
logits   = raw[0] if isinstance(raw, (tuple, list)) else raw
logits   = np.asarray(logits, dtype=np.float32)
probs    = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
y_pred_b = np.argmax(logits, axis=-1)
y_true_b = np.asarray(test_tok['label'], dtype=np.int64)

results.append(evaluate('DistilBERT', y_true_b, y_pred_b, probs))


## 9. Evaluation and Comparison

Here all models are compared using:
- a metric table (Accuracy, Precision, Recall, F1, AUC)
- a bar chart for quick score comparison
- confusion matrices
- ROC curves

For this project, **Recall** is very important because missing a phishing email is risky.


### 9.1 Comparison table (highest F1 first)

In [ ]:
res_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('y_pred', 'y_prob')}
    for r in results
])
res_df = res_df.sort_values('F1', ascending=False).reset_index(drop=True)

display_df = res_df.style.format({
    'Accuracy':  '{:.4f}',
    'Precision': '{:.4f}',
    'Recall':    '{:.4f}',
    'F1':        '{:.4f}',
    'AUC':       '{:.4f}',
}).background_gradient(cmap='Blues', subset=['Accuracy', 'Precision', 'Recall', 'F1', 'AUC'])
display_df


### 9.2 Multi-metric bar chart

In [ ]:

metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
ax = res_df.set_index('Model')[metrics].plot(
    kind='bar', figsize=(11, 5), colormap='viridis', edgecolor='white',
)
plt.title('Model Comparison - Spam / Phishing Detection (Enron corpus)', fontsize=13)
plt.ylim(0.75, 1.01); plt.ylabel('Score'); plt.xlabel('')
plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=1)
plt.tight_layout();
plt.show()


### 9.3 Confusion matrices for every model

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, r in zip(axes, results):
    yt = y_true_b if r['Model'] == 'DistilBERT' else y_test
    cm = confusion_matrix(yt, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
    ax.set_title(r['Model'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout();
plt.show()

### 9.4 ROC overlay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4C9AFF', '#FF6B6B', '#2ECC71', '#F39C12']
for r, color in zip(results, colors):
    if r['y_prob'] is None:
        continue
    yt = y_true_b if r['Model'] == 'DistilBERT' else y_test
    fpr, tpr, _ = roc_curve(yt, r['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{r['Model']} (AUC = {r['AUC']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models')
ax.legend(loc='lower right')
plt.tight_layout();
plt.show()

## 10. Application: `predict_message(text)`

This is the deployable artefact required by the brief - a single function that takes a piece of email text and returns a prediction. It auto-selects the best in-memory model:

```
DistilBERT  >  SVM (TF-IDF)  >  Random Forest (TF-IDF)
```

The classical models reuse `text_cleaning_pipeline()`; DistilBERT receives raw text.


In [ ]:
# Pick the strongest model that was actually trained in this kernel
def _pick_backend():
    if 'model' in globals() and isinstance(model, torch.nn.Module):
        return 'distilbert'
    if 'svm' in globals():
        return 'svm'
    if 'rf' in globals():
        return 'rf'
    raise RuntimeError('No trained model is available in memory.')


LABEL_MAP = {0: 'ham', 1: 'spam'}


def predict_message(text: str, backend: str = 'auto') -> dict:
    """Classify a single email and return prediction + probability.

    Parameters
    ----------
    text : str
        Raw email content (subject + body, or whichever).
    backend : {'auto', 'svm', 'rf', 'distilbert'}
        Which trained model to use. 'auto' picks the strongest available.
    """
    if backend == 'auto':
        backend = _pick_backend()

    if backend in ('svm', 'rf'):
        cleaned = text_cleaning_pipeline(text)
        vec     = tfidf.transform([cleaned])
        clf     = {'svm': svm, 'rf': rf}[backend]
        proba   = float(clf.predict_proba(vec)[0, 1])
        pred    = int(proba >= 0.5)

    elif backend == 'distilbert':
        enc = tokenizer(text, truncation=True, padding=True,
                        max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        prob_vec = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        proba    = float(prob_vec[1])
        pred     = int(prob_vec.argmax())

    else:
        raise ValueError(f'Unknown backend: {backend}')

    return {
        'input':       text[:120] + ('...' if len(text) > 120 else ''),
        'label':       LABEL_MAP[pred],
        'probability': round(proba, 4),
        'model':       backend,
    }


In [ ]:
# Demo on a handful of synthetic emails (3 ham + 3 spam) and present in a table
samples = [
    ('ham',  'Subject: Project status update\n\nHi team, please find attached the Q3 forecast for review.'),
    ('spam', 'Subject: WIN A FREE iPHONE!! Click http://bit.ly/free-iphone now to claim your prize.'),
    ('ham',  'Subject: Lunch tomorrow\n\nWant to grab lunch at the new Thai place around 1pm?'),
    ('spam', 'Subject: Urgent - your account will be closed!! Verify here: www.fake-bank-login.com'),
    ('ham',  'Subject: Re: contract review\n\nVince - I have signed off on section 3. Please proceed with the counter-party.'),
    ('spam', 'CONGRATULATIONS!!! You have been SELECTED as our LUCKY WINNER. Reply STOP to opt out.'),
]

rows = []
for actual, msg in samples:
    r = predict_message(msg)
    rows.append({
        'Actual':      actual,
        'Predicted':   r['label'],
        'Probability': f"{r['probability']:.3f}",
        'Model':       r['model'],
        'Correct':     'yes' if actual == r['label'] else 'no',
        'Message':     msg[:80] + ('...' if len(msg) > 80 else ''),
    })

results_demo = pd.DataFrame(rows)
print('Sample test predictions:')
print(results_demo.to_string(index=False))


## 11. Interactive REPL
Run the cell below to type your own emails. Submit an empty line or `quit` to exit.


In [ ]:
while True:
    text = input('email > ').strip()
    if not text or text.lower() in {'quit', 'exit', ':q'}:
        break
    r = predict_message(text)
    flag = '[SPAM]' if r['label'] == 'spam' else '[HAM]'
    print(f"{flag} p={r['probability']:.3f}  ({r['model']})")
